# Impact of COVID-19 Lockdowns on Air Quality in Bangladesh: Analysis and AQI Forecasting with Support Vector Regression

> **Published at:** IEEE INCET 2023 — 4th International Conference for Emerging Technology, Belgaum, India  
> **Authors:** Mohammed Tahmid Hossain, Afra Hossain, Sabrina Masum Meem, Md Fahad Monir, Md Saef Ullah Miah, Talha Bin Sarwar  
> **Institutions:** Independent University, Bangladesh · AIUB · Universiti Malaysia Pahang

---

## Pipeline

```
CASE Project Dataset (GitHub: TSGreen/bangladesh-air-quality)
  Feb 17, 2014 – Jul 6, 2022 · ~26,000 rows · 10 cities
        │
        ▼  Phase 1 — Pre-processing
Sort by date → Drop NaN (Date:13, Location:49, AQI:641, Category:63)
Replace DNA→NaN→drop (~5000) → Drop Range column → Fix spelling errors
        │
        ▼  Phase 1 — EDA & Comparative Analysis
Split into 3 periods: Pre-COVID (2014-2019) · Lockdown (2020) · Post (2021)
Compute max/min/mean/std per period → Resample monthly/yearly/by location
Location-wise plots: AQImean · AQImin · AQImax + period shading
        │
        ▼  Phase 2 — SVR Forecasting
Train: 2014-02-28 → 2020-12-31
Test : 2021-01-01 → 2022-06-01
MinMaxScaler → SVR(RBF, gamma=0.5, C=10, epsilon=0.05)
Inverse transform → MAPE evaluation (~6.2%)
```

## Table of Contents
1. [Install Dependencies](#1)
2. [Configuration & AQI Table I](#2)
3. [Dataset Download](#3)
4. [Data Pre-processing](#4)
5. [COVID Period Splitting & Statistics](#5)
6. [Resampling & EDA](#6)
7. [Location-Wise AQI Analysis (Figures 3–12)](#7)
8. [SVR Model — Train/Test Split](#8)
9. [SVR Training & Evaluation](#9)
10. [Figure 2 — Actual vs Predicted AQI](#10)
11. [Table II — Sample Results](#11)

<a id='1'></a>
## 1. Install Dependencies

In [ ]:
!pip install -q numpy pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_percentage_error

print(f'Pandas     : {pd.__version__}')
print(f'Scikit-Learn: {__import__("sklearn").__version__}')

<a id='2'></a>
## 2. Configuration & AQI Table I

**Paper Table I:** Approved Air Quality Index (AQI) for Bangladesh

In [ ]:
# 10 cities — Section III-A
CITIES = ['Barishal', 'Chittagong', 'Cumilla', 'Dhaka', 'Gazipur',
          'Khulna', 'Mymensingh', 'Narayanganj', 'Narsingdhi',
          'Rajshahi', 'Sylhet']

# Three COVID periods — Section IV
PERIOD_PRE    = ('2014-02-17', '2019-12-31')   # Before COVID-19
PERIOD_LOCK   = ('2020-01-01', '2020-12-31')   # During lockdown
PERIOD_POST   = ('2021-01-01', '2021-12-31')   # After lockdown

# Train/test split — Section IV
TRAIN_END     = '2020-12-31'
TEST_START    = '2021-01-01'
TEST_END      = '2022-06-01'

# SVR hyperparameters — Section IV (exact values from paper)
SVR_KERNEL    = 'rbf'
SVR_GAMMA     = 0.5
SVR_C         = 10
SVR_EPSILON   = 0.05

# AQI Table I
AQI_CATEGORIES = pd.DataFrame([
    (  0,  50, 'Good',               'Air quality is considered satisfactory with little or no risk.'),
    ( 51, 100, 'Moderate',           'Acceptable; some concern for people sensitive to air pollution.'),
    (101, 150, 'Caution',            'Sensitive groups may have health effects.'),
    (151, 200, 'Unhealthy',          'Everyone may experience health hazards.'),
    (200, 300, 'Very Unhealthy',     'Entire population may experience more serious health effects.'),
    (301, 500, 'Extremely Unhealthy','Everyone likely to be affected by serious health effects.'),
], columns=['AQI_Min','AQI_Max','Level','Health_Implications'])

print('Table I — Approved AQI for Bangladesh:')
print(AQI_CATEGORIES[['AQI_Min','AQI_Max','Level']].to_string(index=False))
print(f'\nSVR hyperparameters: kernel={SVR_KERNEL}, gamma={SVR_GAMMA}, C={SVR_C}, epsilon={SVR_EPSILON}')

<a id='3'></a>
## 3. Dataset Download

**Paper Section III-B:** *"The data was collected from the Clean Air and Sustainable Environment (CASE) project under the Ministry of Environment and Forest of the Government of Bangladesh."*  
**Source:** https://github.com/TSGreen/bangladesh-air-quality  
**Period:** February 17, 2014 – July 6, 2022  
**~26,000 rows · 5 columns:** Date · Location · AQI · AQI Category · Range

In [ ]:
import urllib.request
from pathlib import Path

DATA_PATH = Path('aqi_data.csv')

# Dataset hosted on GitHub (TSGreen/bangladesh-air-quality)
DATASET_URL = ('https://raw.githubusercontent.com/TSGreen/'
               'bangladesh-air-quality/master/data/silver/case/'
               'case_aqi_data.csv')

if not DATA_PATH.exists():
    print('Downloading dataset from GitHub...')
    try:
        urllib.request.urlretrieve(DATASET_URL, DATA_PATH)
        print(f'  Downloaded → {DATA_PATH}')
    except Exception as e:
        print(f'  Download failed: {e}')
        print('  Please download manually from:')
        print(f'  {DATASET_URL}')
        print('  and save as aqi_data.csv')
else:
    print(f'Dataset already present: {DATA_PATH}')

# Load raw
df_raw = pd.read_csv(DATA_PATH)
print(f'\nRaw dataset shape : {df_raw.shape}')
print(f'Columns           : {list(df_raw.columns)}')
print(df_raw.head())

<a id='4'></a>
## 4. Data Pre-processing

**Paper Section III-B — exact steps:**
1. Sort by date
2. Drop NaN: 13 in Date, 49 in Location, 641 in AQI, 63 in AQI Category
3. Replace ~5,000 DNA values in AQI with NaN → drop
4. Drop AQI Range column (insignificant)
5. Fix spelling errors
6. Sort date-wise

In [ ]:
df = df_raw.copy()

print(f'Starting shape: {df.shape}')
print('\nNaN counts per column (before cleaning):')
print(df.isnull().sum())

# Step 1 — Parse dates
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Step 2 — Drop NaN in Date (13), Location (49), AQI (641), AQI Category (63)
# Paper Section III-B specifies exact counts for each column
for col_hint, expected in [('Date',8), ('Location',49), ('AQI',641), ('Category',63)]:
    match = [c for c in df.columns if col_hint.lower() in c.lower()]
    if match:
        before = len(df)
        df = df.dropna(subset=[match[0]])
        print(f'After dropping NaN {col_hint:<12s}: {len(df):,}  '
              f'(removed {before-len(df)}, paper: {expected})')

# Step 3 — Identify AQI column
aqi_col = [c for c in df.columns
            if 'aqi' in c.lower() and 'cat' not in c.lower()][0]

# Step 4 — Replace ~5000 DNA values with NaN then drop
before = len(df)
df[aqi_col] = pd.to_numeric(df[aqi_col].replace('DNA', np.nan), errors='coerce')
df = df.dropna(subset=[aqi_col])
print(f'After replacing DNA→NaN→drop      : {len(df):,}  '
      f'(removed {before-len(df)}, paper: ~5000)')

# Step 5 — Drop AQI Range column (paper: 'not considered and was dropped')
range_cols = [c for c in df.columns if 'range' in c.lower()]
if range_cols:
    df = df.drop(columns=range_cols)
    print(f'Dropped Range column: {range_cols}')

# Step 6 — Fix spelling errors in Location
# Paper mentions fixing spelling errors in the dataset
spelling_fixes = {
    'Comilla'     : 'Cumilla',
    'comilla'     : 'Cumilla',
    'Narsingdi'   : 'Narsingdhi',
    'narsingdi'   : 'Narsingdhi',
    'Narayangonj' : 'Narayanganj',
    'narayangonj' : 'Narayanganj',
}
df['Location'] = df['Location'].replace(spelling_fixes)
print(f'Spelling fixes applied.')

# Step 7 — Sort date-wise for time series
df = df.sort_values('Date').reset_index(drop=True)
df.columns = [c.strip().replace(' ','_') for c in df.columns]
# Re-identify aqi_col after rename
aqi_col = [c for c in df.columns
            if 'aqi' in c.lower() and 'cat' not in c.lower()][0]

print(f'\nFinal shape : {df.shape}')
print(f'Date range  : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Locations   : {sorted(df["Location"].unique())}')
print(df.head())


<a id='5'></a>
## 5. COVID Period Splitting & Statistics

**Paper Section IV:**  
- **Pre-COVID:** Feb 17, 2014 – Dec 31, 2019  
- **During lockdown:** Jan 1, 2020 – Dec 31, 2020  
- **After lockdown:** Jan 1, 2021 – Dec 31, 2021  

*"We calculated the maximum AQI, minimum AQI, mean AQI, and standard deviation for each period"*

In [ ]:
aqi_col = [c for c in df.columns if 'aqi' in c.lower() and 'cat' not in c.lower()][0]

def get_period(df, start, end):
    mask = (df['Date'] >= start) & (df['Date'] <= end)
    return df[mask].copy()

df_pre  = get_period(df, *PERIOD_PRE)
df_lock = get_period(df, *PERIOD_LOCK)
df_post = get_period(df, *PERIOD_POST)

print('AQI Statistics per COVID period:')
print(f'{"Period":<22s} {"N":>7s} {"Max":>8s} {"Min":>8s} {"Mean":>8s} {"Std":>8s}')
print('-' * 60)
for name, subset in [
    ('Pre-COVID (2014-2019)', df_pre),
    ('Lockdown (2020)',       df_lock),
    ('Post-lockdown (2021)',  df_post),
]:
    col = subset[aqi_col]
    print(f'{name:<22s} {len(subset):>7,} {col.max():>8.1f} {col.min():>8.1f} '
          f'{col.mean():>8.1f} {col.std():>8.1f}')

<a id='6'></a>
## 6. Resampling & EDA

**Paper Section IV:** *"We resampled the data at different time intervals to analyze the AQI patterns over different periods.  
It was resampled monthly, yearly, and later by location."*

In [ ]:
df_ts = df.set_index('Date')[[aqi_col]].copy()

# Monthly resample
monthly = df_ts.resample('M').agg(['mean','min','max'])
monthly.columns = ['AQImean','AQImin','AQImax']

# Yearly resample
yearly = df_ts.resample('Y').agg(['mean','min','max'])
yearly.columns = ['AQImean','AQImin','AQImax']

print('Monthly resampled (first 6 rows):')
print(monthly.head(6).round(2))
print(f'\nYearly resampled:')
print(yearly.round(2))

<a id='7'></a>
## 7. Location-Wise AQI Analysis (Figures 3–12)

**Paper Section V-B:** *"line graphs to visualize AQI data for various cities, with yellow bars indicating the minimum AQI, blue bars representing the average AQI, and red bars indicating the maximum AQI. Each figure also highlights the three time periods with a black horizontal scale."*

In [ ]:
def plot_city_aqi(df: pd.DataFrame, city: str, aqi_col: str,
                  save: bool = True):
    """
    Reproduce Figures 3-12 from paper:
    Line plot with AQImean (blue), AQImin (yellow/orange), AQImax (red)
    + shaded regions for 3 COVID periods.
    """
    city_df = df[df['Location'] == city].copy()
    if city_df.empty:
        print(f'  No data for {city}')
        return

    city_ts = city_df.set_index('Date')[[aqi_col]]
    monthly = city_ts.resample('M').agg(['mean','min','max'])
    monthly.columns = ['AQImean','AQImin','AQImax']

    fig, ax = plt.subplots(figsize=(14, 4))
    fig.patch.set_facecolor('#FAFAFA')
    ax.set_facecolor('white')

    # Three period shading
    ax.axvspan(pd.Timestamp(PERIOD_PRE[0]),  pd.Timestamp(PERIOD_PRE[1]),
               alpha=0.08, color='blue',   label='Before Covid')
    ax.axvspan(pd.Timestamp(PERIOD_LOCK[0]), pd.Timestamp(PERIOD_LOCK[1]),
               alpha=0.15, color='orange', label='During Lockdown')
    ax.axvspan(pd.Timestamp(PERIOD_POST[0]), pd.Timestamp(PERIOD_POST[1]),
               alpha=0.08, color='green',  label='After Lockdown')

    # Lines: mean=blue, min=yellow/orange, max=red  (paper colours)
    ax.plot(monthly.index, monthly['AQImean'], color='#1976D2',
            linewidth=1.5, label='AQImean', marker='o', markersize=2)
    ax.plot(monthly.index, monthly['AQImin'],  color='#FFC107',
            linewidth=1.2, label='AQImin',  marker='o', markersize=2)
    ax.plot(monthly.index, monthly['AQImax'],  color='#E53935',
            linewidth=1.2, label='AQImax',  marker='o', markersize=2)

    ax.set_title(f'{city} AQI 2014–2022', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('AQI', fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.3, linestyle='--')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    if save:
        fname = f'fig_aqi_{city.lower().replace(" ","_")}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
    plt.show()
    plt.close()


# Generate Figures 3-12 for all 10 cities
for city in sorted(df['Location'].unique()):
    plot_city_aqi(df, city, aqi_col)

<a id='8'></a>
## 8. SVR Model — Train/Test Split

**Paper Section IV:**
- Training: **2014-02-28 to 2020-12-31**
- Testing:  **2021-01-01 to 2022-06-01**
- MinMaxScaler on AQImean column (range 0, 1)
- SVR: **kernel=RBF, gamma=0.5, C=10, epsilon=0.05** ← exact hyperparameters from paper
- Metric: **MAPE** (~6.2%)

In [ ]:
# Use national daily mean AQI across all locations
daily_mean = (df.set_index('Date')[[aqi_col]]
              .resample('D').mean()
              .dropna()
              .rename(columns={aqi_col: 'AQImean'}))

# Train / test split — exact dates from paper
train = daily_mean.loc['2014-02-28':'2020-12-31']
test  = daily_mean.loc['2021-01-01':'2022-06-01']

print(f'Train: {train.index[0].date()} → {train.index[-1].date()}  '
      f'({len(train):,} days)')
print(f'Test : {test.index[0].date()}  → {test.index[-1].date()}   '
      f'({len(test):,} days)')

# MinMaxScaler — paper: scaled to range (0, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train)
test_scaled  = scaler.transform(test)

print(f'\nScaled train range: [{train_scaled.min():.2f}, {train_scaled.max():.2f}]')
print(f'Scaled test  range: [{test_scaled.min():.2f}, {test_scaled.max():.2f}]')

<a id='9'></a>
## 9. SVR Training & Evaluation

**Paper Section IV:** SVR with RBF kernel, **gamma=0.5, C=10, epsilon=0.05**  
**Paper result: MAPE ≈ 6.2%**

In [ ]:
# Time-series feature engineering for SVR
# Paper Section IV references [14]: 'Time series forecasting with
# support vector regressor' (Microsoft ML-For-Beginners)
# which uses lagged values as features — standard approach for SVR time series.
N_LAGS = 7

def make_lag_features(series: np.ndarray, n_lags: int = 7):
    """Create lag features for time-series SVR (from paper reference [14])."""
    X, y = [], []
    for i in range(n_lags, len(series)):
        X.append(series[i-n_lags:i, 0])
        y.append(series[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = make_lag_features(train_scaled, N_LAGS)
X_test,  y_test  = make_lag_features(test_scaled,  N_LAGS)

# SVR — exact hyperparameters from paper Section IV
svr = SVR(
    kernel  = SVR_KERNEL,    # 'rbf'
    gamma   = SVR_GAMMA,     # 0.5
    C       = SVR_C,         # 10
    epsilon = SVR_EPSILON,   # 0.05
)
svr.fit(X_train, y_train)

# Predict & inverse transform back to original AQI scale
pred_scaled = svr.predict(X_test).reshape(-1, 1)
y_pred      = scaler.inverse_transform(pred_scaled).flatten()
y_actual    = scaler.inverse_transform(y_test.reshape(-1,1)).flatten()

# MAPE — paper metric, result ~6.2%
mape = mean_absolute_percentage_error(y_actual, y_pred) * 100
print(f'SVR: kernel={SVR_KERNEL}, gamma={SVR_GAMMA}, C={SVR_C}, epsilon={SVR_EPSILON}')
print(f'MAPE = {mape:.2f}%  (paper: ~6.2%)')
status = '✅' if mape < 10 else '⚠️'
print(f'{status} Good accuracy' if mape < 10 else f'{status} Check data')


<a id='10'></a>
## 10. Figure 2 — Actual vs Predicted AQI

**Paper Figure 2:** *"Mean values for Actual VS Predicted AQI"* — full time series with both actual and predicted lines.

In [ ]:
# Full time series — predict on training data too for Figure 2
train_pred_sc = svr.predict(X_train).reshape(-1, 1)
train_actual  = scaler.inverse_transform(y_train.reshape(-1,1)).flatten()
train_pred    = scaler.inverse_transform(train_pred_sc).flatten()

train_dates = train.index[N_LAGS:]
test_dates  = test.index[N_LAGS:]

fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('white')

# Actual
ax.plot(train_dates, train_actual, color='#1565C0', linewidth=0.8,
        label='Actual', alpha=0.9)
ax.plot(test_dates,  y_actual,     color='#1565C0', linewidth=0.8, alpha=0.9)

# Predicted
ax.plot(train_dates, train_pred, color='#E91E63', linewidth=0.8,
        label='Predicted', alpha=0.85, linestyle='--')
ax.plot(test_dates,  y_pred,     color='#E91E63', linewidth=0.8,
        alpha=0.85, linestyle='--')

# Mark test region
ax.axvspan(test_dates[0], test_dates[-1], alpha=0.07, color='green',
           label='Test period (2021–2022)')

ax.set_title('Actual VS Predicted Average AQI', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Average AQI', fontsize=11)
ax.legend(fontsize=10, loc='upper right')
ax.grid(alpha=0.3, linestyle='--')
ax.spines[['top','right']].set_visible(False)
ax.annotate(f'MAPE = {mape:.2f}%',
            xy=(0.01, 0.92), xycoords='axes fraction',
            fontsize=11, fontweight='bold', color='#333')

plt.tight_layout()
plt.savefig('fig_actual_vs_predicted_aqi.png', dpi=150,
            bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print('Figure 2 saved.')

<a id='11'></a>
## 11. Table II — Sample Results

**Paper Table II:** Actual vs Predicted Mean AQI (sample rows)

In [ ]:
# Table II — Paper shows IN-SAMPLE predictions from TRAINING data
# Paper Table II dates: 2014-09-02, 2014-09-03, 2015-01-01, etc.
# These are training period predictions, not test predictions.

train_pred_sc = svr.predict(X_train).reshape(-1, 1)
train_actual  = scaler.inverse_transform(y_train.reshape(-1,1)).flatten()
train_pred_vals = scaler.inverse_transform(train_pred_sc).flatten()
train_dates_idx = train.index[N_LAGS:]

# Show first rows that match paper Table II date range (2014-2015)
table2 = pd.DataFrame({
    'Date'        : train_dates_idx[:len(train_actual)][:20],
    'Actual AQI'  : np.round(train_actual[:20], 3),
    'Predicted AQI': np.round(train_pred_vals[:20], 11),
})

print('Table II — Actual vs Predicted Mean AQI (in-sample, training period):')
print(table2.to_string(index=False))
print(f'\nMAPE on test set: {mape:.2f}%  (paper reports ~6.2%)')
print('\nPaper Table II reference values:')
paper_table2 = pd.DataFrame([
    ('2014-09-02', 46.500, 46.5),
    ('2014-09-03', 48.875, 48.875),
    ('2015-01-01', 230.75, 230.75),
    ('2015-01-02', 185.25, 185.12352416),
    ('2015-01-03', 174.25, 174.07473857),
    ('2015-01-07', 210.00, 210.08719749),
    ('2015-01-11', 175.375, 175.20197959),
], columns=['Date','Actual AQI','Predicted AQI'])
print(paper_table2.to_string(index=False))


---
## Key Findings

| Finding | Detail |
|---|---|
| **COVID lockdown reduced AQI** | Highest AQI in 2020 was lower than highest in 2019 or 2021 |
| **Seasonal pattern** | AQI always higher December–March (dry season, low rainfall) |
| **Dhaka most polluted** | Pre-COVID avg AQI 170–190; dropped to 150 during lockdown; rose to 250 after |
| **SVR MAPE ≈ 6.2%** | Good and acceptable accuracy (< 10% threshold) |
| **Inconsistent lockdown** | National avg AQI in 2020 only barely better than 2019 — lockdown not uniformly applied |

---
*Notebook reproducing: "Impact of COVID-19 Lockdowns on Air Quality in Bangladesh: Analysis and AQI Forecasting with Support Vector Regression", IEEE INCET 2023.*